In [1]:
import os
path_figures = os.path.join('..', 'Data', 'Results', 'Figures')

In [ ]:
__author__ = ["Aleksandar Anžel"]
__copyright__ = ""
__credits__ = ["Aleksandar Anžel", "Georges Hattab"]
__license__ = "GNU General Public License v3.0"
__version__ = "1.0.0"
__maintainer__ = "Aleksandar Anžel"
__email__ = "AnzelA@rki.de"
__status__ = "Stable"

# Pie Charts

## Altair/Vega Lite

In [5]:
import altair as alt
import pandas as pd
import os

data = pd.DataFrame({
    'Feature': ['sepal length', 'sepal width', 'petal length', 'petal width'],
    'Value': [0.222, 1.0, 0.0667, 0.0417]
})

pie = alt.Chart(data, width=600, height=600).mark_arc().encode(
    theta=alt.Theta('Value:Q', stack=True), 
    color='Feature:N',
    tooltip=['Feature','Value']
).properties(title="Iris Setosa Feature Distribution")

pie.save("Figures/altair_pie.pdf")


## Plotly

In [ ]:
import plotly.express as px
import pandas as pd
import os

data = pd.DataFrame({
    'Feature': ['sepal length', 'sepal width', 'petal length', 'petal width'],
    'Value': [0.222, 1.0, 0.0667, 0.0417]
})

os.makedirs("Figures", exist_ok=True)
fig = px.pie(data, values='Value', names='Feature', width=600, height=600,
             title="Iris Setosa Feature Distribution")
fig.write_image("Figures/plotly_pie.pdf")

# Radar chart

## Altair/Vega-Lite

In [6]:
import altair as alt
import pandas as pd

# Iris setosa scaled means data
data = pd.DataFrame({
    "Feature": ['sepal length', 'sepal width', 'petal length', 'petal width'],
    "Value": [0.222, 1.0, 0.0667, 0.0417]
})

# To close the polygon, concatenate the first row at the end
data_closed = pd.concat([data, data.iloc[[0]]], ignore_index=True)

chart = alt.Chart(data_closed).mark_line(
    color="crimson",
    interpolate='linear'
).encode(
    theta=alt.Theta("Feature:N", sort=None),
    radius=alt.Radius("Value:Q", scale=alt.Scale(domain=[0,1]))
).properties(
    width=600,
    height=600,
    title="Iris Setosa Feature Profile (Radar Chart)"
) + alt.Chart(data_closed).mark_point(color="crimson", size=100).encode(
    theta=alt.Theta("Feature:N", sort=None),
    radius=alt.Radius("Value:Q")
)

chart.save("Figures/altair_radar.pdf")

## Plotly

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=[0.222,1.0,0.0667,0.0417,0.222],
    theta=['sepal length','sepal width','petal length','petal width','sepal length'],
    fill='toself',
    name='Setosa',
    line_color='crimson'
))
fig.update_layout(
    polar=dict(radialaxis=dict(range=[0,1])),
    width=600, height=600, title="Iris Setosa Feature Profile"
)
fig.write_image("Figures/plotly_radar.pdf")

# Polar Bar Chart

## Altair/Vega-Lite

In [7]:
import altair as alt
import pandas as pd
import math
import os

# Input data
source = pd.DataFrame({
    'feature': ['sepal length', 'sepal width', 'petal length', 'petal width'],
    'value': [0.222, 1.0, 0.0667, 0.0417]
})

# Compute polar positions
n = len(source)
source['angle_index'] = range(n)
source['theta'] = source['angle_index'].apply(lambda i: i * 2 * math.pi / n)

# Manually define best align/baseline based on position
def label_props(theta):
    cos_t, sin_t = math.cos(theta), math.sin(theta)
    align = 'center' if abs(sin_t) < 1e-3 else ('left' if cos_t > 0 else 'right')
    baseline = 'middle' if abs(cos_t) < 1e-3 else ('bottom' if sin_t > 0 else 'top')
    return pd.Series({'align': align, 'baseline': baseline})

source = source.join(source['theta'].apply(label_props))

# Main polar bars
polar_bars = alt.Chart(source).mark_arc(stroke='white').encode(
    theta=alt.Theta("angle_index:O", sort=None),
    radius=alt.Radius("value", scale=alt.Scale(domain=[0, 1])),
    radius2=alt.value(0),
    color=alt.Color("feature:N", legend=None),
    tooltip=['feature:N', 'value:Q']
)

# Grid rings
ring_levels = [0.2, 0.4, 0.6, 0.8, 1.0]
axis_rings = alt.Chart(pd.DataFrame({'ring': ring_levels})).mark_arc(
    stroke='lightgrey', fill=None
).encode(
    theta=alt.value(2 * math.pi),
    radius=alt.Radius('ring')
)

axis_rings_labels = axis_rings.mark_text(
    color='grey', radiusOffset=5, align='left'
).encode(
    text='ring:Q',
    theta=alt.value(math.pi / 4)
)

# Axis lines
axis_lines = alt.Chart(source).mark_arc(stroke='lightgrey', fill=None).encode(
    theta=alt.Theta('theta:Q'),
    radius=alt.Radius(value=1.05),
    radius2=alt.value(0)
)

# Create separate axis label charts for each align/baseline combo
label_charts = []
for align in ['left', 'right', 'center']:
    for baseline in ['top', 'middle', 'bottom']:
        subset = source[(source['align'] == align) & (source['baseline'] == baseline)]
        if not subset.empty:
            label = alt.Chart(subset).mark_text(
                color='grey',
                radiusOffset=100,
                align=align,
                baseline=baseline
            ).encode(
                theta=alt.Theta('theta:Q'),
                radius=alt.Radius(value=2),
                text='feature:N'
            )
            label_charts.append(label)

# Combine all layers
chart = alt.layer(
    axis_rings,
    polar_bars,
    axis_rings_labels,
    axis_lines,
    *label_charts,  # Unpack list of label charts
    title=['Normalized Feature Importance (Iris)', '']
).configure_view(
    stroke=None
).properties(width=600, height=600)

# Save the chart to Figures directory as PNG
chart.save(os.path.join(path_figures, 'altair_polarbar.pdf'))


## Plotly

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Barpolar(
    r=[0.222,1.0,0.0667,0.0417],
    theta=['sepal length','sepal width','petal length','petal width'],
    marker_color=['#440154','#21908C','#FDE725','#F06543']
))
fig.update_layout(width=600, height=600, polar=dict(radialaxis=dict(range=[0,1])),
                  title="Iris Setosa Feature: Polar Bar Chart")
fig.write_image("Figures/plotly_polarbar.pdf")